# 05 · Templated TS: a known geometry onto a fresh molecule

Transfer coordinates with `template=(reference, SMARTS)`. Read an `.xyz` with `rx.read_xyz` first so the
reference carries a molecular graph. A symmetric core or coordinates without a graph need an explicit
`{target_index: reference_index}` map. The mapped atoms are grafted exactly.

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")
SN2 = rx.read_xyz("structures/sn2.xyz", charge=-1)
CORE = "[F].[#6]-[Cl]"
TCORE = rx.match(SN2, CORE)

## One fresh substrate onto the SN2 core

The shared SMARTS maps F, C and Cl between the two molecular graphs. The core is fixed while the propyl
group is searched.

In [ ]:
sub = "[F-].CCCCl"
ens = rx.embed(sub, template=(SN2, CORE)).mc().minimize().prune()
core = rx.match(ens.mol, CORE)
reports = ens.check(frozen=core)
for report in reports.values():
    report.assert_ok()
print(f"1-chloropropane + F-: {ens.n} gate-clean conformers")
xyzrender.render(
    ens.mol,
    no_hy=True,
    ts_bonds=[(core[0] + 1, core[1] + 1), (core[1] + 1, core[2] + 1)],
)

The overlay aligns every conformer on F···C···Cl, showing the fixed core and the searched propyl group.

In [ ]:
xyzrender.render(ens.align(on=CORE).mol, no_hy=True)

## An analogue series on one core

Transfer the same TS onto isopropyl, hexyl, and benzyl chlorides with the same SMARTS.

In [ ]:
for sub in ["[F-].CC(C)Cl", "[F-].CCCCCCCl", "[F-].c1ccccc1CCl"]:
    e = rx.embed(sub, template=(SN2, CORE)).mc().minimize().prune()
    core = rx.match(e.mol, CORE)
    clean = sum(r.ok() for r in e.check(frozen=core).values())
    print(f"  {sub:18s}: {e.n:2d} conformers; {clean}/{e.n} gate-clean")

## A template needn't be a TS: an `Ensemble` works

Any geometry can be a template. The benzaldehyde ring has two traversal directions, so this example maps
its eight atoms explicitly before searching the appended chain.

In [ ]:
BENZALDEHYDE = "O=Cc1ccccc1"
scaffold = rx.embed(BENZALDEHYDE).minimize()
core = tuple(range(8))
ens = rx.embed("O=Cc1ccccc1OCCCC", template=(scaffold, {i: i for i in core}), n=2).minimize()
for report in ens.check(frozen=core).values():
    report.assert_ok()
print(f"benzaldehyde core transferred: {ens.n} gate-clean conformers")
xyzrender.render(ens.align(on=BENZALDEHYDE).mol, no_hy=True)  # core fixed; -OCCCC tail fans out

## Relax the reference without relaxing its TS core

The fixed TS core is not typable by UFF. rxembed relaxes the periphery on a private force-field graph, then
returns the original graph with the core restored.

In [ ]:
reference = rx.minimize(SN2, fix=TCORE)
report = reference.check(frozen=TCORE, reference=SN2)[reference.ids[0]]
report.assert_ok()
print(f"reference TS: {reference.n} conformer; fixed core unchanged; free atoms UFF-relaxed")